#أندري فاركنتين. التنبؤ بأمراض القلب.



### 1. شرح الميزات والبيانات



سنقوم في هذا المشروع بتحليل العوامل التي تساهم في الإصابة بأمراض القلب. تم أخذ البيانات من مستودع التعلم الآلي UCI من [هنا](http://archive.ics.uci.edu/ml/datasets/Heart+Disease)



تم استخدام 14 ميزة:
   
  -- 1. (العمر)     
  
  -- 2. (الجنس) (1 = ذكر، 0 = أنثى)  
  
  -- 3. (cp) نوع الألم في الصدر
    - القيمة 1: الذبحة الصدرية النموذجية
    - القيمة 2: الذبحة الصدرية غير النمطية
    - القيمة 3: الألم غير الزنجاي
    - القيمة 4: بدون أعراض   
    
  -- 4. (trestbps) ضغط الدم أثناء الراحة (بالملليمتر زئبق عند الدخول إلى المستشفى)
    
  -- 5. (كولسترول) مصل الدم بالمجم/ديسيلتر   
  
  -- 6. (fbs) (سكر الدم الصائم > 120 ملجم/ديسيلتر) (1 = صحيح، 0 = خطأ)
  -- 7. (restecg) نتائج تخطيط كهربية القلب أثناء الراحة
  
    --القيمة 0: عادية
    - القيمة 1: وجود شذوذ في موجة ST-T (انقلابات موجة T و/أو ST 
                الارتفاع أو الانخفاض> 0.05 مللي فولت)
    - القيمة 2: إظهار تضخم البطين الأيسر المحتمل أو المؤكد
                وفقا لمعايير إستس
                
  -- 8. (الثلخ) يتم تحقيق الحد الأقصى لمعدل ضربات القلب  
  
  -- 9. (إكسانج) تمرين الذبحة الصدرية (1 = نعم، 0 = لا) 
  
  -- 10. (oldpeak) = اكتئاب ST الناجم عن ممارسة الرياضة مقارنة بالراحة
  
  -- 11. (المنحدر) منحدر تمرين الذروة ST
  
    - القيمة 1: منحدر
    --القيمة 2: مسطحة
    - القيمة 3: الانحدار  
    
  - 12. (كاليفورنيا) عدد الأوعية الكبرى (0-3) الملونة بالدقيق   
  
  -- 13. (ثال) 3 = عادي؛ 6 = عيب ثابت؛ 7 = عيب قابل للعكس   
  
  -- 14. (عدد) تشخيص أمراض القلب (حالة المرض الوعائي)
  
    - خمسة أمراض (0-4)، حيث 0 - الأكثر شيوعا
  


لنقم باستيراد الوحدات الضرورية وقراءة البيانات وتقسيم الميزات إلى عددية وفئوية


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches
matplotlib.style.use('ggplot')
%matplotlib inline
import seaborn as sns

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# data = pd.read_csv('C:/Users/Andrey/Downloads/processed.cleveland.data.txt', sep=",", header=None)
data = pd.read_csv('/home/andrey/Загрузки/processed.cleveland.data.txt', sep=",", header=None)
data.columns = ['age', 'sex', 'pain', 'restbp','chol', 'fbs', 'restecg', 
                'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']

In [ ]:
data.head()

In [ ]:
numer = ['age', 'restbp', 'chol', 'thalach', 'oldpeak']
categ = data.drop(columns=numer).columns.tolist()
categ


### 2. تحليل البيانات الأولية 



دعونا نتعمق في توزيع الميزات


In [ ]:
for i in categ:
    print(i)
    print(data[i].value_counts())
    print(10 * "-")


أوه، هنا لدينا بعض القيم المفقودة. دعونا تنظيف البيانات


In [ ]:
data['ca'] = data['ca'].apply(lambda x: np.nan if str(x) == "?" else x)
data['thal'] = data['thal'].apply(lambda x: np.nan if str(x) == "?" else x)


In [ ]:
data['ca'].fillna(0, inplace = True)
data['thal'].fillna(method='bfill', inplace = True)


In [ ]:
data['ca'].value_counts()

In [ ]:
for i in categ:
    data[i] = data[i].astype("float64")

In [ ]:
data.dtypes


دعونا رسم الميزات.



### 3. تحليل البيانات المرئية الأولية 


In [ ]:
data["num"].value_counts().plot(kind="bar")

plt.title("Target distribution")


الطبقات غير متوازنة، وسوف نأخذ في الاعتبار ذلك في نماذجنا. علاوة على ذلك، هناك ملاحظات قليلة لأمراض القلب المختلفة، وسيكون من الأسهل على الخوارزمية تحديد ما إذا كان الشخص مريضًا بالمرض 0، وسيكون من الصعب التمييز بين الأمراض.


In [ ]:
sns.pairplot(data[numer+["num"]], 
        hue="num", diag_kind="kde")


تبدو جميع الميزات تقريبًا مشابهة للتوزيع الطبيعي. يمكن ملاحظة الحالات الشاذة في العمر وكبار السن عند الأشخاص المصابين بالأمراض 0


In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(16, 16))

for idx, feat in  enumerate(numer+['num']):
    sns.boxplot(x='num', y=feat, data=data, ax=axes[idx // 2, idx % 2])
    axes[idx // 2, idx % 2].legend()
    axes[idx // 2, idx % 2].set_xlabel('num')
    axes[idx // 2, idx % 2].set_ylabel(feat)


لا يمكننا أن نقول أن هناك العديد من الاختلافات بين المجموعات المرصودة من الأشخاص، على الرغم من وجود الكثير من التجاوزات بالنسبة للأشخاص المصابين بالمرض 0، باستثناء متغير العمر


In [ ]:
sns.heatmap(data[numer].corr(), square=True)


معظم المتغيرات المرتبطة هي Restbp والعمر، والذي يرجع إلى فرط التوتر المعتمد على العمر، لكنه لا يزال ضمن المعدل الطبيعي


In [ ]:
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(16, 16))

for idx, feat in enumerate(categ):
    sns.countplot(x=feat, hue='num', data=data, ax=axes[idx // 2, idx % 2]);



الاستنتاجات من الرسم البياني:
- الذكور أكثر عرضة للإصابة بالمرض في مجموعة البيانات لدينا، 
- الألم اللاتماثلي يتوزع بالتساوي بين الطبقات، ويتم الإبلاغ عن أنواع أخرى من الألم في المقام الأول أثناء المرض 0
- نادرا ما يرتفع مستوى السكر في الدم
- هناك شذوذ نادر في موجة ST-T
- تم ممارسة التمارين الرياضية أثناء الذبحة الصدرية في كثير من الأحيان للأشخاص المصابين بأمراض 2-4
- عادة ما يكون ميل الجزء ST من تمرين الذروة مسطحًا لجميع الأمراض باستثناء المرض 0
- يكون العيب قابلاً للشفاء في كثير من الأحيان بالنسبة لجميع الأمراض باستثناء المرض 0


In [ ]:
y = data['num']
X = data.drop(columns=['num'])

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
%%time
tsne = TSNE(random_state=1)
tsne_representation = tsne.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(12,8))
plt.scatter(tsne_representation[:, 0], tsne_representation[:, 1], 
            c = y,  cmap='viridis_r', alpha=.8);


يبدو أن الأشخاص المصابين بالمرض 0 يكون من الأسهل بكثير فصلهم كمجموعة محددة، ومن الصعب العثور على مجموعات بين الأمراض الأخرى


### 4. الرؤى والتبعيات الموجودة 



من EDA السابق وجدنا أن المرض 0 له خصائص أكثر بكثير من الأمراض الأخرى من بياناتنا. ولهذا السبب، في حالتنا، يجب أن نتذكر ضرورة موازنة فئاتنا وأن نكون متشككين إلى حد ما بالنظر إلى النتائج التي سنحصل عليها حول الأمراض من 1 إلى 4، وبالتالي سنحل التصنيف متعدد الفئات



### 5. المعالجة المسبقة للبيانات 



اختر المتغيرات الفئوية لاستخدامها في النماذج


In [ ]:
categ2 = ['sex', 'pain', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']


دعونا نطبق OneHotEncoding على الميزات الفئوية وقياس الميزات الرقمية


In [ ]:
from sklearn.feature_extraction import DictVectorizer as DV

encoder = DV(sparse = False)
cat_hot_x = encoder.fit_transform(X[categ2].T.to_dict().values())

In [ ]:
from sklearn.cross_validation import train_test_split

(X_train_cat_oh,
 X_test_cat_oh) = train_test_split(cat_hot_x, 
                                   test_size=0.3, 
                                   random_state=0, stratify = y)

(X_train_num, 
 X_test_num, 
 y_train, y_test) = train_test_split(X[numer], y, 
                                     test_size=0.3, 
                                     random_state=0, stratify = y)

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

scaler_train = StandardScaler()
scaler_train.fit(X_train_num, y_train)

X_train_num_scaled=scaler_train.transform(X_train_num)
X_test_num_scaled=scaler_train.transform(X_test_num)

x_train=np.hstack((X_train_num_scaled,X_train_cat_oh))
x_test=np.hstack((X_test_num_scaled,X_test_cat_oh))




### 6. اختيار النموذج



نحن نحل مشكلة التصنيف متعدد الفئات. المصنفات التي يدعمها scikit-learn، والتي سنقوم بالتحقق منها هنا:
    
    - متعدد الطبقات بطبيعته:
        - sklearn.tree.DecisionTreeClassifier
        - sklearn.neighbors.KNeighborsClassifier
        - sklearn.ensemble.RandomForestClassifier
        - sklearn.linear_model.RidgeClassifierCV
    - متعدد الفئات كواحد مقابل واحد:
        - sklearn.svm.SVC
    - متعدد الفئات كواحد مقابل الكل:
        -- sklearn.linear_model.LogisticRegression (إعداد multi_class=”ovr”)
        
        
وعلاوة على ذلك، دعونا تشمل XGBoost



### 7. اختيار المقاييس



سنستخدم أيضًا نظام التحقق المتقاطع KFold مع الخلط. علاوة على ذلك، سنقوم بتقسيم بياناتنا إلى مجموعة تدريب واختبار. 
بعد ذلك، سنقوم بتطبيق معلمات النموذج الافتراضية ثم ضبطها باستخدام GridSearch.أما بالنسبة للمقاييس: في حالتنا، من الأفضل الإشارة إلى أن الأشخاص الذين يعانون من مرض أقل خطورة 0 يعانون من أمراض أكثر نادرة، حتى يتمكنوا من إجراء مراقبة إضافية، بدلاً من عدم العثور على أمراض أكثر خطورة. لذا، في هذه الحالة، نحتاج إلى عدد أقل من السلبيات الكاذبة (نقول أن الشخص يتمتع بصحة جيدة نسبيًا، بينما هو ليس كذلك). لذلك نركز على الاستدعاء. لمراعاة ذلك، يمكننا وضع المزيد من الأوزان لاستدعاء المتغير في مقياس f1. في بايثون يمكننا القيام بذلك عن طريق وظيفة fbeta_score. سنقوم أيضًا بتضمين الدقة ودرجة f1 العادية لمعرفة مدى اختلافها عن المقياس المستهدف.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.grid_search import GridSearchCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import NuSVC, SVC
import xgboost as xgb
from sklearn.model_selection import KFold

classifier = [RandomForestClassifier(max_depth=4, criterion='entropy'),
              KNeighborsClassifier(n_neighbors=10),
              xgb.XGBClassifier(learning_rate=0.1, max_depth=5, n_estimators=40, min_child_weight=3),
              RidgeClassifier(random_state=1),
              SVC(random_state=1),
              LogisticRegression(random_state = 0, multi_class='ovr'),
              DecisionTreeClassifier(random_state=1)]

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, fbeta_score
metr = ['accuracy_score', 'roc_auc_score', 'f1_score']

In [ ]:
CV = KFold(n_splits=3, shuffle=True, random_state=1)

In [ ]:
scoress = []
model_name = []
for model in classifier:
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    model_name.append(model.__class__.__name__) 
    scoress.append(({        
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_score_wei': f1_score(y_test, y_pred,average='weighted'),         
        'f_beta': fbeta_score(y_test, y_pred, beta = 1.5, average='weighted')}))

results = pd.DataFrame(data=scoress, columns=['Algorithm', 'accuracy', 'f1_score_wei', 'f_beta'])
results['Algorithm']=model_name

In [ ]:
results


الآن وفقًا لـ f_beta فإن أفضل نموذج هو DecisionTree. وينطبق الشيء نفسه على f1_score، على الرغم من أن النموذج الأكثر دقة هو نموذج Ridge
دعونا ضبط كل شيء



### 8. التحقق من الصحة وتعديل المعلمات الفائقة للنموذج 


In [ ]:
param_grid_rand = {'n_jobs' : [1,5,10,15,20],
              'max_depth' : [5,7,10,15],
              'criterion' : ['gini', 'entropy']}
param_grid_knn = {'n_neighbors':[2,5,10,15],
                  'weights':['uniform', 'distance'],
                  'p': [1, 2]}
param_grid_log = {'penalty' : ['l1', 'l2'],
                  'C' : [0.01, 0.05, 0.1, 0.5, 1, 5, 10]}
param_grid_xgb = {'max_depth' :[2,5,7,10],
                 'learning_rate' : [0.01, 0,1, 1],
                 'gamma' : [0.01, 0.1],
                 'n_estimators' : [20, 30, 40, 50] }

param_grid_ridge = {'alpha' : [0.001, 0.01, 0,1, 1]}
param_grid_svc = {'kernel': ['linear', 'poly', 'rbf'],
                 'degree': [2, 3] ,
                 'decision_function_shape': ['ovo', 'ovr']}

param_grid_tree = {'max_depth' : [3,4,5,7, 10],
                   'min_samples_split' :[2,3,5],
                   'max_features' :['auto', None]}
param_grid = [param_grid_rand, param_grid_knn, param_grid_xgb,param_grid_ridge,  
              param_grid_svc, param_grid_log, 
              param_grid_tree]

In [ ]:
%%time

scores2 = []
model_name = []
for i in enumerate(classifier):
    grid_cv = GridSearchCV(classifier[i[0]], param_grid[i[0]],  cv = 3)
    grid = grid_cv.fit(x_train, y_train.values)
    print(classifier[i[0]], grid_cv.best_params_)
    pred = grid_cv.best_estimator_.predict(x_test)#[:,1]
    model_name.append(classifier[i[0]].__class__.__name__) 
    scores2.append(({        
        'accuracy': accuracy_score(y_test, pred),
        'f1_score_wei': f1_score(y_test, pred, average='weighted'),
        'f_beta': fbeta_score(y_test, pred, beta = 1.5, average='weighted')}))

results2 = pd.DataFrame(data=scores2, columns=['Algorithm', 'f1_score_wei', 'accuracy', 'f_beta'])
results2['Algorithm']=model_name

In [ ]:
results2


دعونا نرسم جميع المقاييس للنماذج المضبوطة


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
f, ax = plt.subplots(figsize=(7, 7))
sns.set(style="whitegrid")

sns.set_color_codes("muted")
sns.barplot(x="accuracy",y = 'Algorithm', data=results2,
            label="accuracy", color="b", alpha= 0.5)

sns.barplot(x="f_beta",y = 'Algorithm', data=results2,
            label="f_beta", color="g", alpha= 0.5)

sns.barplot(x="f1_score_wei", y = 'Algorithm',  data=results2,
            label="f1_score_wei", color="r", alpha = 0.5)


ax.legend(ncol=2, loc="lower right", frameon=True)
ax.set(xlim=(0, 1), ylabel="",
       xlabel="score")
sns.despine(left=True, bottom=True)


كما يمكننا أن نرى أفضل نموذج أداء هو SVC مع المعلمات {'decision_function_shape': 'ovo', 'degree': 2, 'kernel': 'linear'}



### 9. التنبؤ بالعينات الاختبارية أو المحتجزة 



هنا هو التنبؤ بأفضل نموذج


In [ ]:
estimator=SVC(decision_function_shape = 'ovo', degree = 2, kernel='linear')
estimator.fit(x_train, y_train)
pred_new = estimator.predict(x_test)
fbeta_score(y_test, pred_new, beta = 1.5, average='weighted')

In [ ]:
pred_new


### 10. رسم منحنيات التدريب والتحقق من الصحة



لنأخذ أفضل نموذج وننظر إلى منحنيات التعلم


In [ ]:
%%time

RANDOM_SEED = 0
train_sizes, train_scores, test_scores = learning_curve(estimator=SVC(decision_function_shape = 'ovo', 
                                                                      degree = 2, kernel='linear'),
                                                        X=x_train, 
                                                        y=y_train,
                                                        train_sizes=[0.25, 0.5, 0.75, 1.0],
                                                        cv=CV,
                                                        shuffle=True,
                                                        random_state=RANDOM_SEED)

train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

fig = plt.figure(figsize=(20, 5))

plt.xlabel('train_sizes')
plt.ylabel('roc_auc')
# plt.ylim(0.5, 1.01)

plt.plot(train_sizes,
             train_scores_mean,
             label="Training score",
             color="b", marker='o')

plt.plot(train_sizes,
             test_scores_mean, 
             label="CV score",
             color="g", marker='s')

plt.fill_between(train_sizes, 
                 train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, 
                 alpha=0.2, color="b")

plt.fill_between(train_sizes,
                 test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std,
                 alpha=0.2, color="g")


plt.legend(loc="best")
plt.show()


والخبر السار: المنحنيات تقترب من بعضها البعض. الأخبار السيئة: لا شيء يحدث لدرجة السيرة الذاتية. ربما يتطلب هذا النموذج مخططًا محددًا للسيرة الذاتية


In [ ]:
from sklearn.model_selection import StratifiedKFold


CV2 = StratifiedKFold(n_splits=3, random_state=0, shuffle=True)
RANDOM_SEED = 0
train_sizes, train_scores, test_scores = learning_curve(estimator=SVC(decision_function_shape = 'ovo', 
                                                                      degree = 2, kernel='linear'),
                                                        X=x_train, 
                                                        y=y_train,
                                                        train_sizes=[0.25, 0.5, 0.75, 1.0],
                                                        cv=CV2,
                                                        shuffle=True,
                                                        random_state=RANDOM_SEED)

train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

fig = plt.figure(figsize=(20, 5))

plt.xlabel('train_sizes')
plt.ylabel('roc_auc')
# plt.ylim(0.5, 1.01)

plt.plot(train_sizes,
             train_scores_mean,
             label="Training score",
             color="b", marker='o')

plt.plot(train_sizes,
             test_scores_mean, 
             label="CV score",
             color="g", marker='s')

plt.fill_between(train_sizes, 
                 train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, 
                 alpha=0.2, color="b")

plt.fill_between(train_sizes,
                 test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std,
                 alpha=0.2, color="g")


plt.legend(loc="best")
plt.show()


نعم، مع KFold الطبقي الوضع أفضل



### 11. إنشاء ميزات جديدة ووصف هذه العملية 



هل يمكننا تحسين النموذج من خلال إنشاء ميزات جديدة؟ تحتوي البيانات هنا على متغيرات أكثر تصنيفا. في مثل هذه الحالة، دعونا ننشئ ميزات جديدة من خلال التفاعل المتبادل مع الميزات الموجودة. 


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
data_poly_train = poly.fit_transform(x_train)
data_poly_test = poly.transform(x_test)

scaler = StandardScaler()
data_poly_scaled_train = scaler.fit_transform(data_poly_train)
data_poly_scaled_test = scaler.transform(data_poly_test)

In [ ]:

x_train_new=np.hstack((x_train, data_poly_scaled_train))
x_test_new=np.hstack((x_test, data_poly_scaled_test))


In [ ]:
x_train.shape, data_poly_scaled_train.shape, x_train_new.shape

In [ ]:
estimator=SVC(decision_function_shape = 'ovo', degree = 2, kernel='linear')
estimator.fit(x_train_new, y_train)
pred_new = estimator.predict(x_test_new)
fbeta_score(y_test, pred_new, beta = 1.5, average='weighted')

لقد أصبح الأمر أسوأ بكثير، لذا انسى هذا :)



### 12. الاستنتاجات 



لقد جربنا 7 خوارزميات مختلفة لمهمة التصنيف متعددة الفئات. كما نرى أن التمييز بين الأمراض أصعب من معرفة ما إذا كان الشخص مصاباً بأي مرض. في تحليلنا، تبين أن النموذج الأفضل أداءً هو SVC. 
أما بالنسبة لسبل التحسين، فيمكننا أن نرى من منحنيات التعلم أن هناك حاجة لمزيد من البيانات. علاوة على ذلك، يمكن تضمين المزيد من الميزات التي تأخذ في الاعتبار القلب في مجموعة البيانات، مما قد يساعد في بناء خوارزميات أكثر تعقيدًا.
حالة تطبيق مثل هذا النهج واضحة: فهو أتمتة مراقبة المرضى ومساعدة الأطباء في توضيح التشخيص.
آمل أن تجد هذا المشروع مثيرًا للاهتمام، كما فعلت. (^_^)